# Full Pipeline Run

**AI-Based Early Detection and Classification of Foot and Nail Conditions Using Transfer Learning for Rural Healthcare**

Runs the project end to end: fetch data → extract montages → preprocess → train both models.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

Every step is guarded, so re-running a cell that has already completed skips its
work instead of redoing it. If the runtime is recycled part-way through, re-run
from the top — completed steps are restored from Drive rather than recomputed.

**Storage.** All work happens on `/content`, which is fast local disk. Finished
artefacts are packed into single archives and copied to Drive, because Drive
buffers writes and loses tens of thousands of small files when the runtime dies.
Run the `save` cells when you reach them — they are what makes the work survive.

## 1. Setup

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO    = 'https://github.com/gurubasavarajharlapur-jpg/Dissertation_AI_FOOT_NAIL_DISEASE.git'
BRANCH  = 'claude/foot-nail-disease-ai-fyrdsb'
PROJECT = Path('/content/Dissertation_AI_FOOT_NAIL_DISEASE')
DRIVE   = Path('/content/drive/MyDrive/dissertation_foot_nail')

# Clone if absent, pull if present, so this cell is safe to re-run.
if (PROJECT / '.git').is_dir():
    !cd {PROJECT} && git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git pull -q --ff-only
else:
    !git clone -q --branch {BRANCH} {REPO} {PROJECT}

%cd {PROJECT}
!pip install -q -r requirements.txt
!git log --oneline -1

In [ ]:
# An earlier setup symlinked data/ into Drive, which is what lost the data.
# This copies anything still on Drive back to local disk, then removes the links.
!python src/colab_sync.py unlink
!python src/colab_sync.py status

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow', tf.__version__, '| Keras', tf.keras.__version__)
print('GPU:', [g.name for g in gpus] if gpus else
      'NONE — set Runtime > Change runtime type > T4 GPU before the training cells')

## 2. Restore anything already saved

If a previous session got as far as preprocessing or training, this brings it
back and you can skip straight to whichever step is still outstanding.

In [ ]:
!python src/colab_sync.py restore

## 3. What raw data is present?

In [ ]:
IMAGE_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
RAW = PROJECT / 'data' / 'raw'

def count_images(path) -> int:
    path = Path(path)
    if not path.exists():
        return 0
    return sum(1 for f in path.rglob('*')
               if f.is_file() and f.suffix.lower() in IMAGE_EXT)

# Expected counts, from the runs that completed successfully.
EXPECTED = {
    'figshare_nail':       1635,
    'mendeley_foot':       5443,
    'ulcer_fuseg':         1370,
    'figshare_nail_tiles': 18093,
}

print(f"{'folder':<24}{'present':>9}{'expected':>10}")
have = {}
for name, expected in EXPECTED.items():
    n = count_images(RAW / name)
    have[name] = n
    flag = 'ok' if n >= expected * 0.95 else 'MISSING/PARTIAL'
    print(f'{name:<24}{n:>9}{expected:>10}   {flag}')

## 4. Fetch raw data

Each cell skips itself if the data is already there.

In [ ]:
# Figshare onychomycosis: ~2.2 GB
if have['figshare_nail'] < EXPECTED['figshare_nail'] * 0.95:
    !python src/download_data.py --source figshare
    have['figshare_nail'] = count_images(RAW / 'figshare_nail')
else:
    print('figshare_nail already present — skipping')
print('figshare_nail:', have['figshare_nail'])

In [ ]:
# Fallback, only if the cell above returned 0 images or partial zips.
#
# download_data.py resolves the Figshare article and streams each file; if that
# API call is rate-limited or the transfer is cut, wget -c fetches the same
# three files directly by ID and resumes a partial download rather than
# restarting it. The IDs come from Figshare article 5398573.
if have['figshare_nail'] < EXPECTED['figshare_nail'] * 0.95:
    import zipfile

    DL = RAW / '_downloads'
    DL.mkdir(parents=True, exist_ok=True)
    FILES = {
        9302500: 'dataset_A1_thumbnail.zip',
        9302503: 'dataset_A2_thumbnail.zip',
        9302506: 'datasets_B1_B2_C_D_E.zip',
    }
    for file_id, name in FILES.items():
        !wget -c -q --show-progress -O "{DL}/{name}" "https://ndownloader.figshare.com/files/{file_id}"

    # A 0-byte file is a failed transfer reporting success — the exact failure
    # that cost a day earlier. Delete it so -c does not "resume" from nothing.
    for z in sorted(DL.glob('*.zip')):
        if z.stat().st_size == 0:
            print('removing empty:', z.name)
            z.unlink()
        else:
            print(f'{z.stat().st_size / 1e6:>8.0f} MB  {z.name}')

    dest = RAW / 'figshare_nail'
    dest.mkdir(parents=True, exist_ok=True)
    for z in sorted(DL.glob('*.zip')):
        if not zipfile.is_zipfile(z):
            print(f'SKIP {z.name} — not a readable zip')
            continue
        print(f'extracting {z.name} ...')
        with zipfile.ZipFile(z) as zf:
            members = [n for n in zf.namelist()
                       if not n.startswith('__MACOSX/') and '..' not in Path(n).parts]
            zf.extractall(dest, members=members)

    have['figshare_nail'] = count_images(dest)
    print('figshare_nail:', have['figshare_nail'])
else:
    print('figshare_nail present — fallback not needed')


In [ ]:
# FUSeg / AZH foot ulcers: ~600 MB, images only (the labels/ folders are
# segmentation masks and must never enter a classification training set).
import shutil

if have['ulcer_fuseg'] < EXPECTED['ulcer_fuseg'] * 0.95:
    !rm -rf /tmp/ulcer_repo
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/uwm-bigdata/wound-segmentation.git /tmp/ulcer_repo
    !cd /tmp/ulcer_repo && git sparse-checkout set --no-cone '/data/**/images/**'

    SRC  = Path('/tmp/ulcer_repo/data')
    DEST = RAW / 'ulcer_fuseg'
    copied = 0
    for img_dir in sorted(SRC.rglob('images')):
        parts = img_dir.relative_to(SRC).parts
        tag = ('fuseg' if 'Foot Ulcer' in parts[0] else 'medetec') + '_' + parts[-2]
        out = DEST / tag
        out.mkdir(parents=True, exist_ok=True)
        for f in img_dir.iterdir():
            if f.suffix.lower() in IMAGE_EXT:
                shutil.copy2(f, out / f.name)
                copied += 1
        print(f'  {tag:<18} {len(list(out.iterdir())):>5}')
    shutil.rmtree('/tmp/ulcer_repo', ignore_errors=True)
    have['ulcer_fuseg'] = count_images(DEST)
else:
    print('ulcer_fuseg already present — skipping')
print('ulcer_fuseg:', have['ulcer_fuseg'])

In [ ]:
# Mendeley foot images, from the zips uploaded to Drive.
import zipfile

MASKS = PROJECT / 'data' / 'mendeley_masks'

# wound_mask.zip holds SEGMENTATION MASKS, not photographs. They go outside
# data/raw so neither the inspector nor training ever sees them — a binary mask
# counted as a training image is a labelled example of nothing. They are kept
# rather than deleted, being useful for cropping to the wound later.
TARGETS = {
    'normal.zip':     RAW / 'mendeley_foot' / 'Normal',
    'wound_main.zip': RAW / 'mendeley_foot' / 'wound_main',
    'wound_mask.zip': MASKS / 'wound_mask',
}

# Drive's FUSE mount is case-sensitive and the Mendeley export ships
# 'Normal.zip', not 'normal.zip'. Match on the lower-cased name so either
# spelling is found instead of being reported missing.
on_drive = {p.name.lower(): p for p in DRIVE.glob('*.zip')} if DRIVE.exists() else {}

def safe_extract(archive: Path, target: Path) -> int:
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        members = []
        for name in zf.namelist():
            p = Path(name)
            if p.is_absolute() or '..' in p.parts:
                raise RuntimeError(f'unsafe entry in {archive.name}: {name}')
            if name.startswith('__MACOSX/') or Path(name).name.startswith('._'):
                continue
            members.append(name)
        zf.extractall(target, members=members)
    return count_images(target)

if have['mendeley_foot'] < EXPECTED['mendeley_foot'] * 0.95:
    missing = [z for z in TARGETS if z.lower() not in on_drive]
    if missing:
        print('Missing from Drive:', missing)
        print(f'Upload them to {DRIVE}/ and re-run this cell.')
        print('Present:', sorted(p.name for p in on_drive.values()) or '(no zips)')
    else:
        for zip_name, target in TARGETS.items():
            archive = on_drive[zip_name.lower()]
            n = safe_extract(archive, target)
            role = 'MASKS — excluded from training' if 'mask' in zip_name else 'training data'
            print(f'  {archive.name:<16} {n:>5} images   [{role}]')
        have['mendeley_foot'] = count_images(RAW / 'mendeley_foot')
else:
    print('mendeley_foot already present — skipping')
print('mendeley_foot:', have['mendeley_foot'])


## 5. Tile the montage sheets

The Figshare A1/A2 archives are contact sheets — hundreds of nail thumbnails
tiled into one image. Each would otherwise enter training as a single example
containing hundreds of nails. Tiling them is also the only source of **healthy
nail** images, without which the model would classify a healthy nail as fungal.

In [ ]:
if have['figshare_nail_tiles'] < EXPECTED['figshare_nail_tiles'] * 0.95:
    !python src/extract_montages.py
    have['figshare_nail_tiles'] = count_images(RAW / 'figshare_nail_tiles')
else:
    print('tiles already extracted — skipping')
print('figshare_nail_tiles:', have['figshare_nail_tiles'])

In [ ]:
# Everything should be present before preprocessing runs.
print(f"{'folder':<24}{'present':>9}{'expected':>10}")
ready = True
for name, expected in EXPECTED.items():
    n = count_images(RAW / name)
    ok = n >= expected * 0.95
    ready &= ok
    print(f'{name:<24}{n:>9}{expected:>10}   {"ok" if ok else "STILL MISSING"}')
print('\nready for preprocessing:', ready)

## 6. Preprocess

In [ ]:
!python src/preprocessing.py --dry-run

In [ ]:
!python src/preprocessing.py

**Save now.** This is ~250 MB as a single archive, and it is what you would
otherwise have to rebuild after a runtime restart.

In [ ]:
!python src/colab_sync.py save --what processed

## 7. Train MobileNetV2 (primary model)

15 epochs training the head with the backbone frozen, then 25 fine-tuning the
top layers at a 100x lower learning rate, with early stopping.

In [ ]:
assert tf.config.list_physical_devices('GPU'), (
    'No GPU: set Runtime > Change runtime type > T4 GPU, then re-run from cell 1.')

!python src/train.py --model mobilenetv2

In [ ]:
!python src/colab_sync.py save --what models results

## 8. Train ResNet50 (comparison model)

In [ ]:
!python src/train.py --model resnet50

In [ ]:
!python src/colab_sync.py save --what models results

## 9. Where things stand

In [ ]:
!python src/colab_sync.py status
print()
!ls -la models/ results/figures/

---

Send the output of the preprocessing and training cells back to Claude Code, and
Phase 5 (evaluation: confusion matrices, the MobileNetV2 vs ResNet50 comparison,
efficiency metrics and Grad-CAM) can be built against the real results.

## 10. Evaluation (Phase 5)

Reads the test split for the first and only time. Produces the per-class
precision/recall/F1, confusion matrices, model comparison and Grad-CAM figures
that WBS 5.1-5.6 require.

In [ ]:
!python src/evaluate.py

## 11. Calibration (Phase 6a)

Measures whether the confidence numbers can be trusted, fits a temperature on
the **validation** split, and derives the abstention threshold below which the
prototype declines to name a condition.

A softmax output of 0.9 does not generally mean "right nine times in ten" —
deep networks are systematically overconfident (Guo et al., ICML 2017). For a
tool aimed at a health worker with no specialist to consult, a confidently wrong
answer is worse than an admission of uncertainty.

In [ ]:
!python src/calibrate.py

In [ ]:
# Save everything produced so far — cheap, and Colab recycles runtimes.
!python src/colab_sync.py save --what models results

## 12. Confound checks

A four-class model trained on three separately-sourced datasets can reach high
accuracy by recognising *which dataset* an image came from — camera, framing,
background, border padding — rather than the pathology. §2.5.8 of the literature
review names this as the standard weakness of the published work, so the claim
has to be tested rather than asserted. Three independent checks are run here;
Grad-CAM (produced by `evaluate.py` above) is the fourth.


In [ ]:
# Check 1 — border brightness by class and source.
#
# If one class systematically carries a brighter or whiter frame than another,
# a model could separate them on the border alone. This measures the outer 8-px
# frame of a sample from each (class, source) group so any such gap is visible
# as a number rather than assumed absent.
import numpy as np, pandas as pd
from PIL import Image
import sys; sys.path.insert(0, '.')
from src import config

frame = pd.read_csv('data/processed/test.csv')
print(f"{'class':<14}{'source':<22}{'n':>5}{'border bright':>15}{'% near-white':>14}")
for (label, source), grp in frame.groupby(['label', 'source']):
    bright, white = [], []
    for p in grp['path'].head(60):
        a = np.asarray(Image.open(config.PROJECT_ROOT / p).convert('RGB'), dtype=float)
        edge = np.concatenate([a[:8].ravel(), a[-8:].ravel(),
                               a[:, :8].ravel(), a[:, -8:].ravel()])
        bright.append(edge.mean())
        white.append((edge > 230).mean())
    print(f"{label:<14}{source:<22}{len(grp):>5}{np.mean(bright):>15.1f}{np.mean(white) * 100:>13.1f}%")


In [ ]:
# Check 2 — occlusion ablation.
#
# Masking the border and watching accuracy hold is not on its own evidence:
# masking anything removes information, so the run needs an equal-area interior
# control to compare against. ablate_border.py scores four arms — baseline,
# border-masked, an interior control of the same masked area, and centre-masked
# — and reports the drops side by side. A border drop far smaller than the
# equal-area control means the decision does not live in the frame.
!python src/ablate_border.py --model mobilenetv2


In [ ]:
# Check 3 — within-source accuracy.
#
# evaluate.py stores a per-source breakdown in the model's metrics JSON:
# accuracy computed inside each single source, where provenance is constant and
# so cannot carry any signal. Sources holding only one class are marked
# uninformative — a constant prediction scores 100% there. mendeley_foot holds
# both healthy and wound images, which makes it the row that matters.
import json
from pathlib import Path

p = Path('results/metrics/mobilenetv2_metrics.json')
if not p.exists():
    print('run src/evaluate.py first')
else:
    by_source = json.loads(p.read_text())['by_source']
    print(f"{'source':<24}{'n':>6}{'accuracy':>10}   classes")
    for source, row in by_source.items():
        note = '' if row['informative'] else '   (single class — uninformative)'
        print(f"{source:<24}{row['n']:>6}{row['accuracy']:>9.2%}   "
              f"{', '.join(row['classes_present'])}{note}")


In [ ]:
!python src/colab_sync.py save --what results


## 13. Prototype (Phase 6b)

Two tabs:

- **Screening** — upload one photo, get a calibrated prediction, class-based
  guidance and a Grad-CAM overlay. Abstains when unsure.
- **Batch evaluation** — score the test split, or a zip of your own photographs
  organised into folders by class, and compare the two.

Streamlit needs a tunnel to be reachable from your browser in Colab. Run the
next cell, then open the URL it prints and enter the password it shows.

In [ ]:
!pkill -f "streamlit run" 2>/dev/null
import time; time.sleep(2)

# --server.enableCORS false --server.enableXsrfProtection false are required
# behind Colab's port proxy: Streamlit otherwise rejects the proxied origin and
# its websocket never connects, leaving the page stuck on a grey skeleton. Safe
# here — a temporary single-user session, not a public deployment.
!nohup streamlit run src/prototype/app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false --browser.gatherUsageStats false > /content/streamlit.log 2>&1 &

time.sleep(15)
!tail -5 /content/streamlit.log

In [ ]:
# Colab proxies the port out to your browser. Preferred over a tunnel: no
# password, no third-party service. Get a fresh URL after every restart — an
# older one points at the process that has since been killed.
from google.colab.output import eval_js
print("Open this:", eval_js('google.colab.kernel.proxyPort(8501)'))

In [ ]:
# Fallback only, if the proxy URL above will not load.
!npm install -g localtunnel 2>/dev/null | tail -1
print("Tunnel password (paste it on the loca.lt page):")
!curl -s https://loca.lt/mytunnelpassword
!npx --yes localtunnel --port 8501

In [ ]:
# The tunnel password is this machine's public IP.
print("Tunnel password:")
!curl -s https://loca.lt/mytunnelpassword
print("\nOpen the https://....loca.lt URL printed below, then paste that password.\n")
!npx --yes localtunnel --port 8501

**For your viva, run the prototype locally instead.** Tunnels are fine for
development but one more thing to fail on the day:

```bash
git clone --branch claude/foot-nail-disease-ai-fyrdsb <repo-url>
cd Dissertation_AI_FOOT_NAIL_DISEASE
pip install -r requirements.txt
# copy models/ and data/processed/ down from Drive
streamlit run src/prototype/app.py
```

## 14. External validation with your own photographs

The strongest single addition to this dissertation, and nearly free.

Take 20-50 photos on your phone — your own feet and nails, or others' with their
consent. Organise them by class and zip:

```
phone_photos/
  healthy/       IMG_001.jpg ...
  nail_fungal/   IMG_010.jpg ...
  foot_wound/    IMG_020.jpg ...
  foot_ulcer/    IMG_030.jpg ...
```

Upload that zip in the prototype's **Batch evaluation** tab. It scores them and
shows the result beside the test-set figures.

These images come from a different camera, lighting and population than any
training source, so this is the external validation §2.5.8 identifies as missing
from most published work. **If accuracy drops, that is a finding, not a
failure** — it quantifies how far performance on curated clinical photographs
carries to field conditions, and belongs in your results and limitations.

Do not commit the photographs to the public repository, and check whether your
programme requires ethics approval for photographs of people before collecting.